# Watsonx Data Lakehouse Pipeline - Demo

This notebook demonstrates the core components of the real-time IoT Data Lakehouse pipeline:

1. **Schema Registry** - IoT event schemas and validation
2. **Medallion Architecture** - Bronze, Silver, and Gold layer processing
3. **Data Quality** - Expectation suites and automated profiling
4. **Governance** - Lineage tracking and SLA monitoring
5. **Apache Iceberg** - Time-travel and schema evolution utilities

## 1. Setup and Configuration

In [ ]:
import sys
import os
from datetime import datetime, timedelta
import json

# Add project root to path
sys.path.insert(0, os.path.abspath(".."))

print(f"Python: {sys.version}")
print(f"Working dir: {os.getcwd()}")

## 2. Schema Registry - IoT Event Validation

The schema registry defines the structure for all IoT sensor events using both Pydantic models (for validation) and PySpark StructType schemas (for Spark processing).

In [ ]:
from src.ingestion.schema_registry import SensorType, IoTEvent, validate_event

# List all supported sensor types
print("Supported Sensor Types:")
for st in SensorType:
    print(f"  - {st.value}")

In [ ]:
# Create and validate an IoT event
sample_event = {
    "event_id": "evt-demo-001",
    "sensor_id": "sensor-temp-001",
    "sensor_type": "temperature",
    "facility_id": "PLANT-A",
    "timestamp": datetime.utcnow().isoformat(),
    "value": 75.5,
    "unit": "celsius",
    "quality_score": 0.98,
    "is_anomaly": False,
    "metadata": {"firmware": "v2.1"},
}

validated = validate_event(sample_event)
print(f"Validated event: {validated.event_id}")
print(f"Sensor type: {validated.sensor_type.value}")
print(f"Value: {validated.value} {validated.unit}")
print(f"Quality score: {validated.quality_score}")

In [ ]:
# Demonstrate validation failure
invalid_event = {"event_id": "bad-event", "unknown_field": 123}
result = validate_event(invalid_event)
print(f"Invalid event validation result: {result}")
# Returns None for invalid events

## 3. Data Governance - Lineage Tracking

The lineage tracker records every transformation step as data flows through the medallion architecture, enabling full auditability.

In [ ]:
from src.governance.lineage import LineageTracker, LayerType

tracker = LineageTracker()

# Start tracking a pipeline execution
pipeline_id = tracker.start_pipeline()
print(f"Pipeline ID: {pipeline_id}")

In [ ]:
# Record transformation steps through the medallion layers
tracker.record_step(
    name="Kafka Ingestion",
    source_layer=LayerType.SOURCE,
    target_layer=LayerType.BRONZE,
    operation="kafka_consume",
    input_count=10000,
    output_count=10000,
    metadata={"topic": "iot-sensor-events", "partitions": "3"},
)

tracker.record_step(
    name="Deduplication + Cleaning",
    source_layer=LayerType.BRONZE,
    target_layer=LayerType.SILVER,
    operation="deduplicate_clean_enrich",
    input_count=10000,
    output_count=9500,
    metadata={"duplicates_removed": "500", "null_filled": "120"},
)

tracker.record_step(
    name="KPI Aggregation",
    source_layer=LayerType.SILVER,
    target_layer=LayerType.GOLD,
    operation="aggregate_kpis",
    input_count=9500,
    output_count=380,
    metadata={"window_minutes": "15", "facilities": "3"},
)

# Complete the pipeline
completed = tracker.complete_pipeline()
print(f"Pipeline completed with {len(completed.steps)} steps")

In [ ]:
# View the lineage graph
lineage = tracker.get_lineage(pipeline_id=pipeline_id)
print(json.dumps(lineage, indent=2, default=str))

## 4. Data Quality - Expectation Suite

The quality suite validates IoT sensor data against configurable expectations including schema checks, null thresholds, value ranges, and statistical distributions.

In [ ]:
from src.quality.expectations import (
    ExpectationResult,
    QualitySuiteResult,
)

# Demonstrate the quality result model
suite_result = QualitySuiteResult(suite_name="demo_quality_suite")

# Simulate schema check results
required_cols = ["event_id", "sensor_id", "sensor_type", "facility_id", "timestamp", "value", "unit"]
for col in required_cols:
    suite_result.results.append(
        ExpectationResult(name=f"column_exists_{col}", success=True, observed_value=True)
    )

# Simulate range check results
suite_result.results.append(
    ExpectationResult(
        name="range_check_temperature",
        success=True,
        observed_value=0.99,
        details="temperature: 99.0% within [-50.0, 500.0]",
    )
)
suite_result.results.append(
    ExpectationResult(
        name="range_check_vibration",
        success=False,
        observed_value=0.88,
        details="vibration: 88.0% within [0.0, 100.0], 12 out of range",
    )
)

print(f"Suite: {suite_result.suite_name}")
print(f"Overall success: {suite_result.overall_success}")
print(f"Success rate: {suite_result.success_rate:.1%}")
print(f"Total checks: {len(suite_result.results)}")
print(f"Passed: {sum(1 for r in suite_result.results if r.success)}")
print(f"Failed: {sum(1 for r in suite_result.results if not r.success)}")

## 5. Data Profiling

The automated profiler generates distribution statistics, detects null patterns, and identifies low-cardinality columns for optimization.

In [ ]:
from src.quality.profiler import DataProfile, ColumnProfile, DataProfiler

# Create a sample profile manually (in production, this is generated from Spark DataFrames)
profile = DataProfile(name="bronze_iot_data", row_count=50000, column_count=10)

profile.columns = [
    ColumnProfile(name="event_id", dtype="StringType", null_count=0, null_rate=0.0, distinct_count=50000),
    ColumnProfile(name="sensor_id", dtype="StringType", null_count=0, null_rate=0.0, distinct_count=40),
    ColumnProfile(name="sensor_type", dtype="StringType", null_count=0, null_rate=0.0, distinct_count=4),
    ColumnProfile(name="facility_id", dtype="StringType", null_count=0, null_rate=0.0, distinct_count=3),
    ColumnProfile(
        name="value", dtype="DoubleType", null_count=25, null_rate=0.0005, distinct_count=48500,
        stats={"min": -12.5, "max": 485.3, "mean": 75.2, "stddev": 42.1, "median": 72.8},
    ),
    ColumnProfile(name="quality_score", dtype="DoubleType", null_count=150, null_rate=0.003, distinct_count=100,
        stats={"min": 0.1, "max": 1.0, "mean": 0.95, "stddev": 0.08},
    ),
]

print(json.dumps(profile.to_dict(), indent=2, default=str))

In [ ]:
# Generate a quality report from the profile
profiler = DataProfiler(sample_size=10000)
report = profiler.generate_report(profile)

print("Summary:")
print(f"  Total rows: {report['summary']['total_rows']:,}")
print(f"  Total columns: {report['summary']['total_columns']}")
print(f"  High null columns: {report['summary']['high_null_columns']}")
print(f"  Low cardinality columns: {report['summary']['low_cardinality_columns']}")
print(f"\nRecommendations:")
for rec in report["recommendations"]:
    print(f"  - {rec}")

## 6. SLA Monitoring

The SLA monitor tracks data freshness and completeness against configured thresholds, generating alerts when violations occur.

In [ ]:
from src.governance.sla_monitor import SLAMonitor

monitor = SLAMonitor()

# Check freshness - recent data (should be COMPLIANT)
recent = datetime.utcnow() - timedelta(minutes=10)
fresh_check = monitor.check_freshness(recent)
print(f"Freshness check (10 min ago): {fresh_check.status.value}")
print(f"  Lag: {fresh_check.metric_value} min (threshold: {fresh_check.threshold} min)")

# Check freshness - stale data (should be VIOLATED)
stale = datetime.utcnow() - timedelta(hours=2)
stale_check = monitor.check_freshness(stale)
print(f"\nFreshness check (2 hours ago): {stale_check.status.value}")
print(f"  Lag: {stale_check.metric_value} min (threshold: {stale_check.threshold} min)")

In [ ]:
# Check completeness
complete_check = monitor.check_completeness(expected_count=10000, actual_count=9800, layer="bronze")
print(f"Completeness check: {complete_check.status.value}")
print(f"  Completeness: {complete_check.metric_value}% (threshold: {complete_check.threshold}%)")

incomplete_check = monitor.check_completeness(expected_count=10000, actual_count=5000, layer="silver")
print(f"\nCompleteness check: {incomplete_check.status.value}")
print(f"  Completeness: {incomplete_check.metric_value}% (threshold: {incomplete_check.threshold}%)")

In [ ]:
# View SLA dashboard data
dashboard = monitor.get_dashboard_data()
print(f"Total checks performed: {dashboard['total_checks']}")
print(f"Current freshness status: {dashboard['current_status']['freshness']}")
print(f"Current completeness status: {dashboard['current_status']['completeness']}")
print(f"Thresholds: {json.dumps(dashboard['thresholds'], indent=2)}")

## 7. Iceberg Table Management

Apache Iceberg provides time-travel queries, schema evolution, and snapshot management. Below shows the API (requires a running Spark session with Iceberg catalog configured).

In [ ]:
from src.lakehouse.iceberg_utils import IcebergManager

# Note: These operations require a running SparkSession with Iceberg
# Below demonstrates the API structure

print("IcebergManager API:")
print("  - create_table(table_name, schema_df, partition_columns, properties)")
print("  - time_travel(table_name, snapshot_id=None, as_of_timestamp=None)")
print("  - list_snapshots(table_name) -> list of snapshot metadata")
print("  - get_table_history(table_name) -> change history")
print("  - evolve_schema(table_name, add_columns, rename_columns)")
print("  - expire_snapshots(table_name, older_than) -> reclaim storage")
print("  - compact_data_files(table_name) -> optimize file sizes")

print("\nExample time-travel query:")
print('  df = iceberg.time_travel("db.bronze_events", as_of_timestamp="2024-01-15T10:00:00")')
print("\nExample schema evolution:")
print('  iceberg.evolve_schema("db.bronze_events", add_columns={"region": "string"})')

## 8. Configuration Management

The pipeline uses a centralized configuration system combining environment variables (`.env`) and YAML settings (`config/settings.yaml`) through Pydantic Settings.

In [ ]:
import yaml

config_path = os.path.join("..", "config", "settings.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

print("Lakehouse layer configuration:")
for layer in ["bronze", "silver", "gold"]:
    layer_cfg = config["lakehouse"][layer]
    print(f"\n  {layer.upper()} Layer:")
    print(f"    Path: {layer_cfg['path']}")
    print(f"    Partitions: {layer_cfg['partition_columns']}")

print("\nQuality expectations:")
for sensor, thresholds in config["quality"]["expectations"].items():
    print(f"  {sensor}: range=[{thresholds.get('min_value')}, {thresholds.get('max_value')}], null_threshold={thresholds.get('null_threshold')}")

print(f"\nSLA thresholds:")
sla = config["governance"]["sla"]
print(f"  Freshness: {sla['freshness_threshold_minutes']} min")
print(f"  Completeness: {sla['completeness_threshold_percent']}%")

---

## Summary

This demo covered the core components of the Watsonx Data Lakehouse Pipeline:

| Component | Module | Description |
|---|---|---|
| Schema Registry | `src.ingestion.schema_registry` | Pydantic + PySpark schema validation |
| Bronze Layer | `src.lakehouse.bronze` | Raw data ingestion with partitioning |
| Silver Layer | `src.lakehouse.silver` | Dedup, cleaning, enrichment |
| Gold Layer | `src.lakehouse.gold` | KPI aggregation, anomaly scoring |
| Iceberg Manager | `src.lakehouse.iceberg_utils` | Time-travel, schema evolution |
| Quality Suite | `src.quality.expectations` | Data quality expectation checks |
| Profiler | `src.quality.profiler` | Automated data profiling |
| Lineage Tracker | `src.governance.lineage` | Data lineage graphs |
| SLA Monitor | `src.governance.sla_monitor` | Freshness and completeness SLAs |

For full pipeline execution with Spark, Kafka, and Iceberg, use `docker-compose up -d` to start the infrastructure.